In [1]:
import json
from pathlib import Path
from tqdm import tqdm
import multiprocessing as mp
from langdetect import detect, LangDetectException, DetectorFactory
from itertools import islice

DetectorFactory.seed = 0  # make langdetect deterministic-ish

In [2]:
def _detect_english_from_line(line: str):
    line = line.strip()
    if not line:
        return None  # skip blank lines

    try:
        rec = json.loads(line)
    except json.JSONDecodeError:
        return None  # skip invalid JSON

    text = rec.get("preprocessed_content", "")
    if not isinstance(text, str) or text.strip() == "":
        rec["is_english"] = False
    else:
        try:
            rec["is_english"] = (detect(text) == "en")
        except LangDetectException:
            rec["is_english"] = False

    return json.dumps(rec, ensure_ascii=False)


def print_first_jsonl_safe(path, n=100, max_field_len=300):
    print(f"\n--- First {n} entries of {path} (safe preview) ---")
    with open(path, "r", encoding="utf-8", errors="replace") as f:
        for i, line in enumerate(islice(f, n), 1):
            try:
                rec = json.loads(line)
            except json.JSONDecodeError:
                print(f"\nEntry {i}: Invalid JSON")
                continue

            print(f"\n================ Entry {i} ================")
            for k, v in rec.items():
                if isinstance(v, str) and len(v) > max_field_len:
                    print(f"{k}: {v[:max_field_len]}... (len={len(v)})")
                else:
                    print(f"{k}: {v}")

In [3]:
def add_english_column_jsonl_parallel(
    input_jsonl,
    output_jsonl,
    limit_rows=None,
    workers=None,
    chunk_size=500
):
    input_jsonl = Path(input_jsonl)
    output_jsonl = Path(output_jsonl)
    output_jsonl.parent.mkdir(parents=True, exist_ok=True)

    if workers is None:
        workers = max(1, mp.cpu_count() - 1)

    written = 0
    skipped = 0

    with input_jsonl.open("r", encoding="utf-8", errors="replace") as fin, \
         output_jsonl.open("w", encoding="utf-8") as fout, \
         mp.Pool(processes=workers) as pool:

        # Iterator of lines (optionally limited)
        if limit_rows is not None:
            def limited_iter(f, n):
                for _ in range(n):
                    line = f.readline()
                    if not line:
                        break
                    yield line
            lines_iter = limited_iter(fin, limit_rows)
        else:
            lines_iter = fin

        print(f"Using {workers} worker processes (chunksize={chunk_size})")

        for result in tqdm(
            pool.imap(_detect_english_from_line, lines_iter, chunksize=chunk_size),
            desc="Detecting English (parallel)"
        ):
            if result is None:
                skipped += 1
                continue
            fout.write(result + "\n")
            written += 1

    print(f"\nDone.")
    print(f"Wrote rows: {written:,}")
    print(f"Skipped (blank/invalid JSON): {skipped:,}")
    print(f"Output: {output_jsonl.resolve()}")

In [4]:
out_file = "preprocessed_text_english_.jsonl"

add_english_column_jsonl_parallel(
    input_jsonl="preprocessed_text.jsonl",
    output_jsonl=out_file,
    limit_rows=None,     # set 1000 for quick test
    workers=8,           # adjust to your CPU cores (or None)
    chunk_size=500       # tune: 200–2000 depending on workload
)

Using 8 worker processes (chunksize=500)


Detecting English (parallel): 11403638it [55:25, 3429.42it/s] 


Done.
Wrote rows: 11,403,638
Skipped (blank/invalid JSON): 0
Output: /home/darknet/2026-01-27_201419_domain_with_snapshots/preprocessed_text_english_.jsonl


In [7]:
print_first_jsonl_safe("preprocessed_text_english_.jsonl", n=100)


--- First 100 entries of preprocessed_text_english_.jsonl (safe preview) ---

================ Entry 1 ================
created_at: 1585293617000
darknet_type: torv2
domain_id: 163884
domain_url: http://hack3q3l3szn2aib.onion
file_size: 10104
html_file_location: htmls/00/8c/008c6311d24df8c8e190c79e3d43874e.html
page_id: 808271
path: /
path_hash: 42099b4af021e53fd8fd4e056c2568d7c2e3ffa8
snapshot_hash: 008c6311d24df8c8e190c79e3d43874e
snapshot_id: 24417808
tags: Cybercrime,English,Hacking,Service Provider,Social Engineering,Social Media
title: Hacker | Cyber Crime Solution
html_code: <!DOCTYPE html>
<html lang="en">
	<head>
		<title>Hacker | Cyber Crime Solution</title>
		<style>
			body {background:url("tiles.jpg") black;text-align:center;}
			body, input, textarea, button, legend {
			font-family: "Lucida Grande",Helvetica,Arial,Verdana,sans-serif;
			color: #333;
			... (len=10104)
raw_content: 
preprocessed_content: 
is_english: False

================ Entry 2 ================
creat